# P vs NP – snadno ověřit, těžko najít

* **P** – úlohy, které umíme *vyřešit* v polynomiálním čase ($n$, $n^2$, $n^3$, ...): třídění, nejkratší cesta, ...
* **NP** – úlohy, u nichž lze navržené řešení (*certifikát*) *ověřit* v polynomiálním čase.
  Najít ho může trvat exponenciálně dlouho ($2^n$, $n!$) – pokud je známo.
* **NP-úplné** – nejtěžší úlohy v NP. Každou úlohu z NP lze převést (*redukovat*) na kteroukoli z nich,
  takže rychlý algoritmus pro jednu by dal rychlý algoritmus pro všechny.

Zda $P = NP$ je nejslavnější otevřená otázka informatiky (Millennium Prize, 1 000 000 dolarů).
Skoro všichni věří, že $P \neq NP$ – ale nikdo to nedokázal.

Past pro programátora: NP-úplné úlohy mívají **zadání na jeden řádek**, které vypadá triviálně.
V tomto notebooku se podíváme na několik z nich – vždy se stejným vzorem:

> *zkontrolovat* řešení je pár řádků rychlého kódu – *najít* ho trvá déle než stáří vesmíru.

*Jak číst kód:* čtěte komentáře; přesná syntaxe Pythonu je vedlejší.

### Přesná definice (stojí za to ji mít správně)

Problém je **NP-úplný**, když splňuje *obě* tyto podmínky zároveň:

1. **je v NP** – navržené řešení lze ověřit v polynomiálním čase, a
2. **je NP-těžký** – každý problém z NP se na něj dá v polynomiálním čase převést (*redukovat*).

Samotná podmínka 2 dává **NP-těžký** problém; ten v NP být vůbec nemusí.
NP-úplné mohou být jen *rozhodovací* problémy (odpověď ano / ne), protože jen u nich existuje certifikát, který lze ověřit.

**TSP jako příklad:**

| otázka | třída | proč |
|---|---|---|
| „Existuje okružní trasa kratší než 500 km?“ | **NP-úplná** | v NP: certifikát je trasa, sečteme její úseky; NP-těžká: převede se na ni problém Hamiltonovy kružnice |
| „Najdi nejkratší okružní trasu.“ | **NP-těžká**, není známo, že by byla v NP | není to otázka ano/ne – a neznáme krátký certifikát, že trasa je *nejkratší*: museli bychom vyloučit všechny kratší |

V praxi jsou obě verze stejně těžké (několika otázkami „kratší než L?“ najdeme optimum), proto se zkráceně říká
„TSP je NP-úplný“. Přesně: rozhodovací verze je NP-úplná, optimalizační verze je NP-těžká.
Extrémní příklad NP-těžkého problému mimo NP je problém zastavení, který nelze vyřešit vůbec.

In [ ]:
import itertools                 # kombinace, permutace, kartézské součiny
import math
import random
from time import perf_counter    # stopky

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx            # grafy

YEAR = 365.25 * 24 * 3600
AGE_OF_UNIVERSE = 13.8e9 * YEAR      # sekund od velkého třesku


def human_time(seconds: float) -> str:
    """Převede sekundy na čitelný text: '3.2 ms', '1.7 days', '6.4 thousand years', ..."""
    units = [("ns", 1e-9), ("µs", 1e-6), ("ms", 1e-3), ("s", 1), ("min", 60), ("h", 3600),
             ("days", 86400), ("years", YEAR), ("thousand years", 1e3 * YEAR),
             ("million years", 1e6 * YEAR), ("billion years", 1e9 * YEAR)]
    if seconds > 1e6 * 1e9 * YEAR:
        return f"{seconds / (1e9 * YEAR):.1e} billion years"
    for name, size in reversed(units):
        if seconds >= size:
            return f"{seconds / size:,.1f} {name}"
    return f"{seconds / 1e-9:.2f} ns"

## 0. Kam patří zkušenost z TSP? Třídy růstu na jeden pohled

V notebooku TSP jsme potkali $(n-1)!$ a $n^2 \cdot 2^n$. Postavme je vedle ostatních běžných rychlostí růstu.
Tabulka předpokládá **1 mikrosekundu na elementární operaci** (zhruba jeden krok Pythonu; C je asi 100× rychlejší,
což – jak jsme viděli – posune obrázek jen o pár položek).

| třída | typický příklad z tohoto předmětu |
|---|---|
| $\log n$ | půlení intervalu v seřazeném seznamu |
| $n$ | test Eulerova tahu (spočítej stupně), ověření certifikátu |
| $n \log n$ | třídění, Dijkstra / A* s haldou (na hranu) |
| $n^2$ | heuristika nejbližšího souseda, průchod 2-opt, Prim na hustém grafu |
| $n^3$ | všechny nejkratší cesty (Floyd–Warshall), násobení matic (naivní) |
| $2^n$ | subset sum hrubou silou, 3-barvení je $3^n$ |
| $n^2 \cdot 2^n$ | Held–Karp (přesný TSP) |
| $n!$ | TSP hrubou silou, Hamiltonova kružnice hrubou silou |

Vše nad čarou $2^n$ je **polynomiální** – zdvojnásobení vstupu znásobí práci konstantou (2, 4, 8).
Vše od $2^n$ dolů je **exponenciální** nebo horší – *přidání jedné položky* znásobí práci konstantou nebo víc.

In [ ]:
def log10_time_to_text(log10_seconds: float) -> str:
    """Čitelný čas z log10(sekund). Počítáme s logaritmy, protože 1000! je pro obyčejné číslo příliš velké."""
    if log10_seconds < 25:
        return human_time(10 ** log10_seconds)
    return f"10^{log10_seconds - math.log10(YEAR):.0f} years"


MICROSECOND = 1e-6
# Pro každou třídu růstu: funkce, která vrátí log10(počet operací) pro velikost vstupu n
growth = {
    "log n":     lambda n: math.log10(math.log2(n)),
    "n":         lambda n: math.log10(n),
    "n log n":   lambda n: math.log10(n * math.log2(n)),
    "n^2":       lambda n: 2 * math.log10(n),
    "n^3":       lambda n: 3 * math.log10(n),
    "2^n":       lambda n: n * math.log10(2),
    "n^2 * 2^n": lambda n: 2 * math.log10(n) + n * math.log10(2),
    "n!":        lambda n: math.log10(math.factorial(n)),
}
sizes = [10, 20, 50, 100, 1000]

# čas = operace x 1 µs   ->   log10(čas) = log10(operace) + log10(1e-6)
table = pd.DataFrame({f"n = {n}": [log10_time_to_text(f(n) + math.log10(MICROSECOND)) for f in growth.values()] for n in sizes},
                     index=pd.Index(growth.keys(), name="operací"))
print("Doba běhu při 1 µs na operaci (stáří vesmíru = 13,8 miliardy let):")
table

In [ ]:
# Totéž jako obrázek: n od 2 do 100, logaritmická svislá osa
ns = np.arange(2, 101)
plt.figure(figsize=(9, 5.5))
for name, f in growth.items():
    plt.plot(ns, [10 ** (f(n) + math.log10(MICROSECOND)) for n in ns], label=name)
plt.yscale("log")
for label, value in {"1 sekunda": 1, "1 hodina": 3600, "1 rok": YEAR, "stáří vesmíru": AGE_OF_UNIVERSE}.items():
    plt.axhline(value, ls=":", color="gray"); plt.text(3, value * 2, label, color="gray")
plt.ylim(1e-7, 1e40)
plt.xlabel("velikost vstupu n"); plt.ylabel("čas [s] při 1 µs na operaci"); plt.grid(True, which="both", alpha=0.3)
plt.legend(loc="upper left", ncol=2)
plt.title("Třídy růstu: polynomiální (dole) vs. exponenciální a faktoriálové (nahoře)")
plt.show()

Dvě věci, které si z obrázku odnést:

* Na logaritmické ose se polynom ohýbá *dolů*, exponenciála je přímka, faktoriál se ohýbá *nahoru*.
* U polynomiálních algoritmů koupí 1000× rychlejší počítač 1000× (n), 32× (n²) nebo 10× (n³) větší vstup.
  U $2^n$ koupí **+10 položek**, u $n!$ **+2–3 položky**. To je praktický význam otázky P vs NP níže.

## 1. Součet podmnožiny (subset sum)

**Zadání:** je dán seznam čísel a cíl; existuje podmnožina, jejíž součet se rovná cíli?

*Příklad:* faktury `[12, 7, 25, 3, 18, 31]`, bankovní převod `43` – které faktury byly zaplaceny?
(Účetní tohle opravdu řeší.)

In [ ]:
def verify_subset_sum(numbers: list[int], target: int, certificate: tuple[int, ...]) -> bool:
    """Polynomiální – O(n): jen sečti vybraná čísla a porovnej. `certificate` = pozice vybraných čísel."""
    return sum(numbers[i] for i in certificate) == target


def solve_subset_sum(numbers: list[int], target: int):
    """Exponenciální – zkouší až 2^n podmnožin: nejdřív všechny velikosti 0, pak 1, 2, ..."""
    n = len(numbers)
    for r in range(n + 1):                                     # velikost podmnožiny r = 0, 1, ..., n
        for subset in itertools.combinations(range(n), r):     # každý výběr r pozic
            if sum(numbers[i] for i in subset) == target:      # našli jsme -> hotovo
                return subset
    return None                                                # vyzkoušeno vše -> řešení není


invoices = [12, 7, 25, 3, 18, 31]
solution = solve_subset_sum(invoices, 43)
print("Řešení:", solution, "->", [invoices[i] for i in solution])
print("Ověřeno:", verify_subset_sum(invoices, 43, solution))

### Jak roste čas?

Abychom změřili nejhorší případ, použijeme instanci, která **nemá** řešení (všechna čísla sudá, cíl lichý),
takže hrubá síla musí vyzkoušet všech $2^n$ podmnožin.

In [ ]:
rng = random.Random(0)
times = {}
for n in range(4, 21):
    numbers = [2 * rng.randint(1, 10 ** 6) for _ in range(n)]   # n náhodných SUDÝCH čísel
    t0 = perf_counter()
    solve_subset_sum(numbers, 1)          # lichý cíl -> nemožné -> zkontroluje se všech 2^n podmnožin
    times[n] = perf_counter() - t0
    print(f"n = {n:2d}: {2 ** n:>9,} podmnožin, {times[n]:.4f} s")

n_max = max(times)
time_per_subset = times[n_max] / 2 ** n_max      # čas na jednu podmnožinu z největšího měření

plt.figure(figsize=(8, 5))
plt.semilogy(list(times), list(times.values()), "s-", label="změřeno")
ns = np.arange(4, 100)
plt.semilogy(ns, [2.0 ** n * time_per_subset for n in ns], "--", label="extrapolace 2^n")
for label, value in {"1 hodina": 3600, "1 rok": YEAR, "stáří vesmíru": AGE_OF_UNIVERSE}.items():
    plt.axhline(value, ls=":", color="gray"); plt.text(5, value * 2, label, color="gray")
plt.xlabel("počet položek n"); plt.ylabel("čas [s]"); plt.grid(True, which="both", alpha=0.3); plt.legend()
plt.title("Subset sum hrubou silou")
plt.show()

n_universe = next(n for n in ns if 2.0 ** n * time_per_subset > AGE_OF_UNIVERSE)
print(f"Hrubá síla překročí stáří vesmíru při n = {n_universe} položkách.")
for n in (30, 40, 50, 60, 80, 100):
    print(f"n = {n:3d}: {human_time(2.0 ** n * time_per_subset)}")

In [ ]:
# ...zatímco ověření zůstává okamžité i pro milion čísel
numbers = [rng.randint(1, 10 ** 9) for _ in range(1_000_000)]      # milion náhodných čísel
certificate = tuple(rng.sample(range(len(numbers)), 1000))         # navržená odpověď: 1000 z nich
target = sum(numbers[i] for i in certificate)                      # (cíl nastavíme tak, aby odpověď seděla)

t0 = perf_counter()
ok = verify_subset_sum(numbers, target, certificate)
print(f"n = 1 000 000: ověřeno = {ok} za {1000 * (perf_counter() - t0):.2f} ms")

(Subset sum má algoritmus dynamického programování se složitostí $O(n \cdot \text{cíl})$.
Vypadá polynomiálně, ale *cíl* je číslo, ne velikost: s 64bitovými čísly je to zase $2^{64}$ kroků.
Takovým úlohám se říká *slabě* NP-úplné – kvůli tomu padl batohový kryptosystém a je to častá zkoušková otázka.)

## 2. Stejný graf, dvě otázky: Euler vs. Hamilton

Obě otázky chtějí uzavřenou procházku grafem, která projde *vše právě jednou*:

* **Eulerův tah** – každou **hranu** právě jednou (sedm mostů v Königsbergu, trasy pluhů, sekvenování DNA).
  Euler 1736 dokázal: existuje právě tehdy, když je graf souvislý a každý vrchol má sudý stupeň. To je $O(V + E)$ – **P**.
* **Hamiltonova kružnice** – každý **vrchol** právě jednou. Nikdo nezná nic lepšího než (chytře) zkoušet permutace – **NP-úplné**.
  TSP je její optimalizační verze: Hamiltonova kružnice s minimální celkovou délkou.

In [ ]:
def has_euler_circuit(G: nx.Graph) -> bool:
    """Eulerova věta: souvislý + každý vrchol má sudý počet hran. Stačí spočítat."""
    return nx.is_connected(G) and all(degree % 2 == 0 for _, degree in G.degree())


def count_hamiltonian_cycles(G: nx.Graph) -> int:
    """Hrubá síla: vyzkoušej každé pořadí vrcholů a zkontroluj, že sousední jsou spojené hranou."""
    nodes = list(G)
    first, count = nodes[0], 0
    for perm in itertools.permutations(nodes[1:]):          # (n-1)! pořadí ostatních vrcholů
        cycle = (first, *perm, first)                       # uzavři kružnici
        if all(G.has_edge(a, b) for a, b in zip(cycle, cycle[1:])):   # každá sousední dvojice musí být hrana
            count += 1
    return count


rows = []
for n in range(5, 11):
    G = nx.random_regular_graph(4, n, seed=1)      # každý vrchol má stupeň 4 -> Eulerův tah existuje
    t0 = perf_counter(); euler = has_euler_circuit(G); t_euler = perf_counter() - t0
    t0 = perf_counter(); hamilton = count_hamiltonian_cycles(G); t_hamilton = perf_counter() - t0
    rows.append({"vrcholů": n, "Eulerův tah existuje": euler, "čas Euler [ms]": 1000 * t_euler,
                 "Hamiltonových kružnic": hamilton, "čas Hamilton [ms]": 1000 * t_hamilton})

# Eulerův test na obrovském grafu: 20 000 vrcholů, 200 000 hran
big = nx.gnp_random_graph(20_000, 0.001, seed=1)
t0 = perf_counter(); has_euler_circuit(big); t_big = perf_counter() - t0
print(f"Eulerův test na grafu s {big.number_of_nodes():,} vrcholy a {big.number_of_edges():,} hranami: {1000 * t_big:.0f} ms")
pd.DataFrame(rows).set_index("vrcholů")

## 3. Jedno slovo dělá rozdíl: nejkratší vs. nejdelší cesta

**Nejkratší** cesta z jednoho rohu mřížky do druhého: Dijkstra, polynomiální (minulé cvičení).

**Nejdelší** *jednoduchá* cesta (žádný vrchol dvakrát) mezi týmiž rohy: NP-těžká. Jediná obecná metoda je vypsat cesty,
a počet cest z rohu do rohu v mřížce $m \times m$ exploduje: 2, 12, 184, 8 512, 1 262 816, 575 780 564, ...

In [ ]:
rows = []
for m in range(2, 6):
    grid = nx.grid_2d_graph(m, m)                    # mřížka m x m vrcholů
    s, t = (0, 0), (m - 1, m - 1)                    # protilehlé rohy

    t0 = perf_counter()
    shortest = nx.shortest_path_length(grid, s, t)   # prohledávání do šířky: rychlé
    t_short = perf_counter() - t0

    t0 = perf_counter()
    n_paths, longest = 0, 0
    for path in nx.all_simple_paths(grid, s, t):     # vypiš KAŽDOU cestu bez opakování vrcholů ...
        n_paths += 1
        longest = max(longest, len(path) - 1)        # ... a pamatuj si nejdelší
    t_long = perf_counter() - t0

    rows.append({"mřížka": f"{m}x{m}", "nejkratší cesta": shortest, "čas [ms]": 1000 * t_short,
                 "nejdelší cesta": longest, "vypsaných cest": n_paths, "čas [s]": t_long})

time_per_path = t_long / n_paths
for m, n_paths in [(6, 575_780_564), (7, 789_360_053_252), (8, 3_266_598_486_981_642)]:
    print(f"{m}x{m}: {n_paths:,} cest -> zhruba {human_time(n_paths * time_per_path)}")
pd.DataFrame(rows).set_index("mřížka")

## 4. Snadní a těžcí sourozenci – úlohy, které zná každý

Mnoho těžkých úloh má snadného sourozence: zadání zní skoro stejně, změní se jedna podmínka –
a z minut je věčnost. K jejich rozpoznání nepotřebujete učebnicové názvy (pro zájemce jsou v závorkách):

| úloha z běžného života | snadná verze – P | těžká verze – NP-úplná / NP-těžká | co se změnilo |
|---|---|---|---|
| Rozvoz | nejrychlejší cesta ze skladu na **jednu** adresu (nejkratší cesta – Dijkstra, A*) | nejkratší trasa přes **všech 20** adres (TSP) | jeden cíl → všechny cíle |
| Seřazení | seřadit adresy podle abecedy (třídění) | seřadit je do nejlepšího pořadí rozvozu (opět TSP) | řazení podle klíče → podle vzájemných vzdáleností |
| Obchůzka čtvrti | projít **každou ulici** právě jednou – pošťák, sněžný pluh (Eulerův tah) | projít **každou křižovatku** právě jednou (Hamiltonova kružnice) | ulice → křižovatky |
| Účetnictví | sečíst všechny faktury | najít, **které** faktury dají dohromady platbu 43 (subset sum) | sečíst vše → vybrat podmnožinu |
| Nakládka dodávky | naložit co nejcennější náklad, když zboží jde **dělit** – písek, kapaliny (zlomkový batoh) | totéž s **nedělitelnými** krabicemi (batoh 0/1) | dělitelné → nedělitelné |
| Rozvrh zkoušek | rozdělit zkoušky do **dvou** termínů tak, aby nikdo neměl dvě naráz (2-barvení) | totéž do **tří** termínů (3-barvení) | 2 → 3 |
| Sestavení týmů | rozdělit lidi do **dvojic**, které spolu vycházejí (párování) | rozdělit je do **trojic**, které spolu vycházejí (3-dimenzionální párování) | 2 → 3 |
| Pokládka kabelu | propojit vesnice co nejlevněji (minimální kostra) | totéž, ale smíte navíc postavit **nové rozbočovače** kdekoli (Steinerův strom) | víc volnosti, těžší úloha |
| Výlet | nejkratší cesta přes park | nejdelší procházka, která žádné místo nenavštíví dvakrát (nejdelší jednoduchá cesta) | nejkratší → nejdelší |
| Sudoku | zkontrolovat vyplněnou mřížku | vyplnit prázdnou mřížku (obecné sudoku n²×n²) | ověřit → vyřešit |
| Počítání | vynásobit dvě 300místná čísla | rozložit 600místné číslo na dva prvočinitele (faktorizace – stojí na ní RSA; v NP, asi ne NP-úplná) | dopředu → pozpátku |

Vzor: nepatrná změna v zadání, obrovská změna v obtížnosti. Některé těžké verze jsou v běžném životě dost malé na to,
aby se stejně vyřešily (sudoku 9×9, dodávka s deseti krabicemi) – NP-těžkost mluví o nejhorším případě, když velikost roste.

**Proč to v praxi vadí?**

* Rozvrhování (rozvrhy, směny, CNC zakázky), plánování tras, řezné plány, návrh čipů, skládání proteinů – vše NP-těžké.
* Přesný rychlý algoritmus *nenajdete*. Místo něj: heuristiky, aproximační algoritmy (Christofides: nejvýš 1,5× optimum),
  lokální prohledávání (2-opt, simulované žíhání, genetické algoritmy), nebo solvery (SAT / MIP / constraint programming),
  které jsou v nejhorším případě exponenciální, ale na typických instancích rychlé.
* Kryptografie na tom staví obráceně: *chceme*, aby faktorizace a diskrétní logaritmus byly těžké.

## Cvičení: 3-barvení grafu

**Zadání:** obarvěte vrcholy grafu třemi barvami tak, aby žádná hrana nespojovala dva vrcholy stejné barvy
(přidělování registrů v překladačích, přidělování frekvencí vysílačům, rozvrh zkoušek).

1. Naprogramujte `verify_colouring` – polynomiální verifikátor.
2. Naprogramujte `solve_3_colouring` hrubou silou přes všech $3^n$ obarvení (`itertools.product`).
3. Otestujte na Petersenově grafu (3-barvitelný) a na $K_4$ (není 3-barvitelný). Změřte, jak roste čas s `nx.cycle_graph(n)`.

In [ ]:
def verify_colouring(G: nx.Graph, colouring: dict) -> bool:
    """Vrátí True, pokud žádná hrana nespojuje dva vrcholy stejné barvy. `colouring` = {vrchol: barva}."""
    # YOUR CODE HERE
    return False


def solve_3_colouring(G: nx.Graph):
    """Vrátí platné obarvení {vrchol: barva} nebo None, pokud graf není 3-barvitelný."""
    nodes = list(G)
    # YOUR CODE HERE: for colours in itertools.product(range(3), repeat=len(nodes)): ...
    return None


for name, G in [("Petersenův graf", nx.petersen_graph()), ("K4", nx.complete_graph(4))]:
    t0 = perf_counter()
    colouring = solve_3_colouring(G)
    print(f"{name}: {'obarvitelný' if colouring else 'není 3-barvitelný / neimplementováno'} "
          f"({perf_counter() - t0:.3f} s)", "ověřeno:" if colouring else "", verify_colouring(G, colouring) if colouring else "")

if (colouring := solve_3_colouring(nx.petersen_graph())):
    nx.draw(nx.petersen_graph(), node_color=[colouring[v] for v in nx.petersen_graph()], cmap="Set1", with_labels=True)